<a href="https://colab.research.google.com/github/shikhadiwakar/lightweight-cyber-threat-detector/blob/main/Copy_of_01_dataset_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lightweight Cybersecurity Threat Detection

**Research Project:**
Lightweight Machine Learning-Based Cybersecurity Threat Detection for Resource-Constrained Hardware

**Research Question:**
How much detection performance can be retained while reducing the computational and memory requirements of a cybersecurity threat detection model?

**Author:** Anush Jindal

**Notebook 1 of 3 — Dataset Exploration**

This notebook is Stage 2 ("Dataset") of the project. The goal here is only to **load and understand** the data — no cleaning, no modeling yet. By the end you should be able to answer: how many rows/columns are there, what type is each column, are there missing values or duplicates, and what does the target label look like?

## 1. Import the libraries we need

These are almost all pre-installed in Google Colab already. If any `import` below fails with `ModuleNotFoundError`, run `!pip install <package_name>` in a new cell above it and try again.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Makes plots show up nicely inside the notebook
%matplotlib inline

print("Libraries imported successfully.")
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

**Expected output:** three lines confirming the import worked and printing version numbers (exact numbers will vary — that's fine).

**Troubleshooting:** if you see `ModuleNotFoundError: No module named 'seaborn'` (or similar), run this in a new cell: `!pip install seaborn` then re-run the import cell.

## 2. Load the dataset

Before running the cell below, make sure you've uploaded **one** CICIDS2017 day-file (see `data/README.md` for exactly which file and where to get it). In Colab:
- Click the folder icon on the left sidebar
- Click the upload icon and choose your CSV file
- Wait for the upload to finish (there's a progress bar)

Then update the filename in the cell below to match exactly what you uploaded.

In [ ]:
# CHANGE THIS to the exact filename of the file you uploaded
FILENAME = "Wednesday-workingHours.pcap_ISCX.csv"

df = pd.read_csv(FILENAME)

print("Loaded successfully.")
print("Type of df:", type(df))

**Expected output:**
```
Loaded successfully.
Type of df: <class 'pandas.core.frame.DataFrame'>
```

**Troubleshooting:**
- `FileNotFoundError` — the filename doesn't match exactly what's in your Colab file browser (check capitalization, spaces, and the `.csv` extension). Click the folder icon and copy the exact name.
- Loading feels slow or the notebook seems stuck — CICIDS2017 day-files can be tens or hundreds of MB; give it a minute. If it's still stuck after several minutes, your file may be larger than expected — double check you downloaded a single day's file, not the whole dataset archive.
- `UnicodeDecodeError` — some CICIDS2017 mirrors use a different encoding. Try: `pd.read_csv(FILENAME, encoding="latin1")`.

## 3. Shape — how many rows and columns?

`df.shape` returns `(number_of_rows, number_of_columns)`. This is usually the very first thing to check about any new dataset.

In [ ]:
df.shape

**Expected output format:** a tuple like `(225745, 79)` — meaning 225,745 rows and 79 columns. **Your actual numbers will be different** depending on which day's file you downloaded — write down your real numbers in `experiments/experiment_notes.md`, don't assume these example numbers.

## 4. First look at the actual data

`df.head()` shows the first 5 rows so you can see real values, not just column names.

In [ ]:
df.head()

## 5. Column names

Some CICIDS2017 file versions have a **leading space** before column names (e.g. `" Label"` instead of `"Label"`) — this trips up a lot of beginners because `df["Label"]` will fail with a `KeyError` even though you can see a column called `Label`. Printing the exact column list makes this visible immediately.

In [ ]:
print(f"Number of columns: {len(df.columns)}\n")
for col in df.columns:
    print(repr(col))  # repr() shows quotes so leading/trailing spaces are visible

**Optional cleanup (safe to do here — this just renames columns, it doesn't change any data):** if you spot stray whitespace in your column names, strip it now so the rest of the notebook is easier to write:
```python
df.columns = df.columns.str.strip()
```

In [ ]:
# Remove leading/trailing spaces from column names, if any
df.columns = df.columns.str.strip()
print("Column names cleaned.")

## 6. Data types & structure

`df.info()` shows, for every column: how many non-null values it has, and its data type (`int64`, `float64`, `object` for text, etc.). This is how you spot columns that *should* be numbers but were loaded as text.

In [ ]:
df.info()

## 7. Summary statistics

`df.describe()` gives count, mean, standard deviation, min, max, and quartiles for every **numeric** column. Look for anything suspicious — e.g. a `max` value of `inf` (infinity) is common in CICIDS2017's rate-based columns (like bytes/sec) and needs handling in the next notebook.

In [ ]:
df.describe()

## 8. Missing values

`df.isnull().sum()` counts, per column, how many cells are `NaN` (missing/empty).

In [ ]:
missing = df.isnull().sum()

# Only show columns that actually have missing values, sorted worst-first
missing_only = missing[missing > 0].sort_values(ascending=False)
print(f"Columns with missing values: {len(missing_only)} out of {len(df.columns)}\n")
missing_only

**Also check for infinite values** — `isnull()` does **not** catch `inf`/`-inf`, which CICIDS2017 is known to contain in some rate columns (e.g. Flow Bytes/s can become infinite when Flow Duration is 0).

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
inf_counts = np.isinf(numeric_df).sum()
inf_only = inf_counts[inf_counts > 0].sort_values(ascending=False)
print(f"Numeric columns containing infinite values: {len(inf_only)}\n")
inf_only

**Note down** (in `experiments/experiment_notes.md`) which columns had missing or infinite values and roughly how many — you'll need this in `02_preprocessing.ipynb`. Don't fix anything yet in this notebook; this notebook is for *understanding* only.

## 9. Duplicate rows

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Number of fully duplicated rows: {num_duplicates}")
print(f"That's {num_duplicates / len(df) * 100:.2f}% of all rows")

## 10. The target label column

This is the column that says whether each row is normal traffic or an attack (and which kind). In most CICIDS2017 files it's called `Label` (after the whitespace-stripping we did in Step 5) — but always confirm with `df.columns` rather than assuming.

In [ ]:
LABEL_COLUMN = "Label"  # change this if your file uses a different column name

assert LABEL_COLUMN in df.columns, f"'{LABEL_COLUMN}' not found. Actual columns: {list(df.columns)}"

print("Raw counts:")
print(df[LABEL_COLUMN].value_counts())
print("\nAs percentages:")
print((df[LABEL_COLUMN].value_counts(normalize=True) * 100).round(2))

**Expected output shape (illustrative only — do not copy these exact numbers):**
```
BENIGN              440031
DoS Hulk             23012
PortScan             15880
DDoS                 12802
...
```
The real class names and counts depend entirely on which day's file you downloaded — read `research/attack_categories.md` once you have your real label list, to understand what each label you actually see means.

**What to look for:** is `BENIGN` the large majority? (It usually is — this is a class imbalance you'll need to keep in mind later, when "99% accuracy" might just mean "always predicting BENIGN.") Are any attack classes so rare (a handful of rows) that they'll be hard to learn from at all?

In [ ]:
plt.figure(figsize=(10, 5))
df[LABEL_COLUMN].value_counts().plot(kind="bar")
plt.title("Class distribution")
plt.ylabel("Number of flows")
plt.xlabel("Label")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

## 11. Understand what each label actually means

Open `research/attack_categories.md` and, for every label you saw printed in Step 10 above, confirm you understand what that attack type actually is. If you see a label not covered there (some file versions have slightly different sub-labels, e.g. `Web Attack � Brute Force`), add a short description for it to that file.

**Do not merge/rename classes in this notebook.** That's a modeling decision for later — for now we're only trying to understand the data as-is.

## 12. Summary — record your observations

Before moving to `02_preprocessing.ipynb`, open `experiments/experiment_notes.md` and add an entry answering:

- Which exact file did you use?
- How many rows and columns? (from Step 3)
- Which columns had missing values, and roughly how many? (Step 8)
- Which columns had infinite values? (Step 8)
- How many duplicate rows? (Step 9)
- What does the class distribution look like — which classes are common, which are rare? (Step 10)

This record is what makes your cleaning decisions in the next notebook defensible — you're cleaning based on what you actually observed, not guesswork.